In [3]:
import torch
import os
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"VRAM in use: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"PID: {os.getpid()}")

VRAM in use: 0.00 GB
PID: 1606481


In [4]:
import torch
import gc
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"Tensors on CUDA:")
for obj in gc.get_objects():
    try:
        if torch.is_tensor(obj) and obj.is_cuda:
            print(f"  {obj.shape} {obj.dtype} {obj.device} ({obj.numel()*obj.element_size()/1e6:.1f}MB)")
    except:
        pass

VRAM: 0.00 GB
Tensors on CUDA:


/ix/cs2770_2026s/abn80/conda/envs/medvlm/lib/python3.11/site-packages/torch/__init__.py:1164: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)


In [5]:
import torch
import torch.nn.functional as F
from torch.amp import autocast
from transformers import LlavaForConditionalGeneration, AutoProcessor
from datasets import load_from_disk

exec(open("diffusion_pretrain.py").read().split('if __name__')[0])

device = torch.device("cuda")
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# 1. Load model
print("\n[1/3] Loading model...")
full_model = LlavaForConditionalGeneration.from_pretrained(
    "../models/llava-1.5-7b-hf", torch_dtype=torch.float32, low_cpu_mem_usage=True,
)
vision_model = full_model.model.vision_tower.vision_model
diffusion_vit = DiffusionViT(vision_model, num_timesteps=1000).to(device)
del full_model, vision_model
torch.cuda.empty_cache()
print(f"  VRAM after model: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# 2. Load one batch
print("\n[2/3] Loading batch...")
processor = AutoProcessor.from_pretrained("../models/llava-1.5-7b-hf")
ds = load_from_disk("../data/medtrinity-demo/hf_dataset")
dataset = MedTrinityDiffusionDataset(ds, processor)
loader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True, num_workers=0)
pixel_values = next(iter(loader)).to(device)
print(f"  Batch: {pixel_values.shape}")

# 3. Run 5 training steps (clean loop — no stale activations)
print("\n[3/3] Training steps...")
noise_schedule = CosineNoiseSchedule(num_timesteps=1000, device=device)
timestep_sampler = StratifiedTimestepSampler(num_timesteps=1000)
optimizer = torch.optim.AdamW(diffusion_vit.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler(enabled=True)

diffusion_vit.train()
for i in range(5):
    optimizer.zero_grad()
    with autocast(device_type="cuda", enabled=True):
        x_0 = diffusion_vit.get_patch_embeddings(pixel_values)
        noise = torch.randn_like(x_0)
        timesteps = timestep_sampler.sample(16, device=device)
        x_t = noise_schedule.add_noise(x_0, noise, timesteps)
        predicted = diffusion_vit.forward_encoder(x_t, timesteps)
        target = noise_schedule.get_v_target(x_0, noise, timesteps)
        loss = F.mse_loss(predicted, target)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    print(f"  Step {i+1}: loss={loss.item():.4f} | VRAM={torch.cuda.memory_allocated()/1e9:.2f} GB")

    # Verify grads on first step
    if i == 0:
        cg = diffusion_vit.embeddings.patch_embedding.weight.grad
        eg = diffusion_vit.encoder.layers[0].self_attn.q_proj.weight.grad
        print(f"    Conv2d grad norm: {cg.norm().item():.4f}")
        print(f"    Encoder L0 grad norm: {eg.norm().item():.4f}")

print(f"\n  Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
print("\nSanity check PASSED!")

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB

[1/3] Loading model...


Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


  VRAM after model: 1.23 GB

[2/3] Loading batch...


Loading dataset from disk:   0%|          | 0/18 [00:00<?, ?it/s]

  Batch: torch.Size([16, 3, 336, 336])

[3/3] Training steps...
  Step 1: loss=0.6458 | VRAM=5.13 GB
    Conv2d grad norm: 0.6289
    Encoder L0 grad norm: 0.0038
  Step 2: loss=0.6338 | VRAM=5.14 GB
  Step 3: loss=0.6288 | VRAM=5.14 GB
  Step 4: loss=0.6149 | VRAM=5.14 GB
  Step 5: loss=0.6017 | VRAM=5.14 GB

  Peak VRAM: 15.72 GB

Sanity check PASSED!
